
# Flight Delay Prediction Using Machine Learning

## Client Overview
An airline company wants to predict flight delays using airline, route, date, and weather-related data to improve customer satisfaction and operational planning.

## Project Objective
Build and compare classification models that predict whether a flight will be **Delayed** or **On Time**.

## Models
- Logistic Regression
- Random Forest Classifier

## Evaluation Metrics
- Accuracy
- Precision
- Recall
- F1-score
- Confusion Matrix
- ROC Curve
- Classification Report


In [1]:

# Import libraries
import os
import warnings
warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    roc_curve,
    roc_auc_score
)

pd.set_option("display.max_columns", None)
RANDOM_STATE = 42


## 1. Load the Dataset

In [4]:
import os

print(os.getcwd())       # Shows the current working folder
print(os.listdir())      # Lists all files in that folder

c:\Users\Mirza Shareef Baig\OneDrive\Desktop\Skybrisk\month4_Flight_Delay_Prediction_Project
['flight_data_2024.csv', 'flight_data_2024_data_dictionary.csv', 'Flight_Delay_Prediction.ipynb', 'Model_Evaluation_Report.md', 'README.md', 'requirements.txt']


In [7]:

# Update this path if your CSV file has a different name.
DATA_PATH = DATA_PATH = "flights.csv"
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Dataset not found at '{DATA_PATH}'. "
        "Place the Kaggle CSV file in the same folder as this notebook "
        "or update DATA_PATH."
    )

df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
display(df.head())


FileNotFoundError: Dataset not found at 'flights.csv'. Place the Kaggle CSV file in the same folder as this notebook or update DATA_PATH.

In [ ]:

# Review columns, data types, and missing values
print(df.info())

missing_summary = (
    df.isnull()
      .mean()
      .mul(100)
      .sort_values(ascending=False)
      .to_frame("Missing_Percentage")
)

display(missing_summary.head(20))


## 2. Data Preparation

In [ ]:

# Standardize column names for easier processing
df.columns = (
    df.columns.astype(str)
      .str.strip()
      .str.replace(" ", "_", regex=False)
      .str.replace("-", "_", regex=False)
)

print(df.columns.tolist())


In [ ]:

# Drop columns with more than 50% missing values
missing_threshold = 0.50
high_missing_cols = [
    col for col in df.columns
    if df[col].isnull().mean() > missing_threshold
]

print("Columns dropped because of excessive missing values:")
print(high_missing_cols)

df = df.drop(columns=high_missing_cols)
print("Shape after dropping high-missing columns:", df.shape)


In [ ]:

# Convert FlightDate into useful date features
date_candidates = [
    "FlightDate", "Flight_Date", "FL_DATE", "Date", "flight_date"
]

date_col = next((col for col in date_candidates if col in df.columns), None)

if date_col:
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df["Flight_Weekday"] = df[date_col].dt.dayofweek
    df["Flight_Month"] = df[date_col].dt.month
    df["Flight_Day"] = df[date_col].dt.day
    df = df.drop(columns=[date_col])
    print(f"Date features created from: {date_col}")
else:
    print("No FlightDate column was found. Check the dataset column names.")


In [ ]:

# Create the target variable
# The code supports several common airline-delay dataset formats.

target_col = None

# Case 1: Existing binary target
binary_target_candidates = [
    "Delayed", "IsDelayed", "Delay_Status", "Flight_Status",
    "DEP_DEL15", "ArrDel15", "ARR_DEL15"
]

for col in binary_target_candidates:
    if col in df.columns:
        target_col = col
        break

# Case 2: Create target from arrival or departure delay minutes
if target_col is None:
    delay_minute_candidates = [
        "ArrDelay", "ARR_DELAY", "Arrival_Delay",
        "DepDelay", "DEP_DELAY", "Departure_Delay",
        "ArrivalDelay", "DepartureDelay"
    ]

    delay_col = next(
        (col for col in delay_minute_candidates if col in df.columns),
        None
    )

    if delay_col:
        # Standard aviation convention: delayed if 15 or more minutes late
        df["Delayed"] = (pd.to_numeric(df[delay_col], errors="coerce") >= 15).astype(int)
        target_col = "Delayed"
        print(f"Target created from '{delay_col}' using a 15-minute threshold.")

if target_col is None:
    raise ValueError(
        "Could not identify a delay target. "
        "Please set target_col manually using your dataset's delay column."
    )

# Convert text targets to binary where needed
if df[target_col].dtype == "object":
    normalized = df[target_col].astype(str).str.strip().str.lower()
    mapping = {
        "delayed": 1,
        "delay": 1,
        "late": 1,
        "yes": 1,
        "1": 1,
        "on time": 0,
        "on-time": 0,
        "ontime": 0,
        "no": 0,
        "0": 0
    }
    df[target_col] = normalized.map(mapping)

df = df.dropna(subset=[target_col])
df[target_col] = df[target_col].astype(int)

print("Target column:", target_col)
print(df[target_col].value_counts())
print(df[target_col].value_counts(normalize=True))


In [ ]:

# Remove columns that may cause target leakage or are unlikely to generalize
possible_leakage_cols = [
    "ArrDelay", "ARR_DELAY", "Arrival_Delay",
    "DepDelay", "DEP_DELAY", "Departure_Delay",
    "ArrivalDelay", "DepartureDelay",
    "Cancelled", "CancellationCode", "Diverted",
    "ActualElapsedTime", "AirTime",
    "ArrTime", "DepTime",
    "WheelsOff", "WheelsOn"
]

leakage_cols_to_drop = [
    col for col in possible_leakage_cols
    if col in df.columns and col != target_col
]

print("Potential leakage columns removed:", leakage_cols_to_drop)
df = df.drop(columns=leakage_cols_to_drop)


In [ ]:

# Define features and target
X = df.drop(columns=[target_col])
y = df[target_col]

# Drop identifiers with extremely high cardinality
high_cardinality_cols = [
    col for col in X.select_dtypes(include="object").columns
    if X[col].nunique(dropna=True) > 1000
]

print("High-cardinality columns removed:", high_cardinality_cols)
X = X.drop(columns=high_cardinality_cols)

categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
numeric_features = X.select_dtypes(include=["number"]).columns.tolist()

print("Categorical features:", categorical_features)
print("Numeric features:", numeric_features)
print("Final feature count:", X.shape[1])


## 3. Train/Test Split and Preprocessing

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)


In [ ]:

# Preprocessing pipeline
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)


## 4. Model Building

In [ ]:

logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ))
    ]
)

random_forest_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=300,
            max_depth=None,
            min_samples_split=5,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ]
)

models = {
    "Logistic Regression": logistic_pipeline,
    "Random Forest": random_forest_pipeline
}


In [ ]:

# Train both models
for model_name, model in models.items():
    print(f"Training {model_name}...")
    model.fit(X_train, y_train)

print("Model training completed.")


## 5. Model Evaluation

In [ ]:

evaluation_results = []

for model_name, model in models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    result = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1_Score": f1_score(y_test, y_pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test, y_prob)
    }
    evaluation_results.append(result)

results_df = pd.DataFrame(evaluation_results).sort_values(
    by="ROC_AUC",
    ascending=False
)

display(results_df)


In [ ]:

# Classification reports
for model_name, model in models.items():
    y_pred = model.predict(X_test)
    print("=" * 70)
    print(model_name)
    print("=" * 70)
    print(classification_report(
        y_test,
        y_pred,
        target_names=["On Time", "Delayed"],
        zero_division=0
    ))


In [ ]:

# Confusion matrices
for model_name, model in models.items():
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["On Time", "Delayed"]
    )
    disp.plot(values_format="d")
    plt.title(f"Confusion Matrix - {model_name}")
    plt.tight_layout()
    plt.show()


In [ ]:

# ROC curves
plt.figure(figsize=(8, 6))

for model_name, model in models.items():
    y_prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc_score = roc_auc_score(y_test, y_prob)

    plt.plot(fpr, tpr, label=f"{model_name} (AUC = {auc_score:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", label="Random Classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 6. Select and Save the Best Model

In [ ]:

# Select the model with the highest ROC-AUC score
best_model_name = results_df.iloc[0]["Model"]
best_model = models[best_model_name]

MODEL_PATH = "flight_delay_model.pkl"
joblib.dump(best_model, MODEL_PATH)

print(f"Best model: {best_model_name}")
print(f"Saved model to: {MODEL_PATH}")


In [ ]:

# Save evaluation metrics
results_df.to_csv("model_evaluation_results.csv", index=False)

with open("classification_reports.txt", "w", encoding="utf-8") as report_file:
    for model_name, model in models.items():
        y_pred = model.predict(X_test)
        report_file.write("=" * 70 + "\n")
        report_file.write(model_name + "\n")
        report_file.write("=" * 70 + "\n")
        report_file.write(
            classification_report(
                y_test,
                y_pred,
                target_names=["On Time", "Delayed"],
                zero_division=0
            )
        )
        report_file.write("\n\n")

print("Evaluation files saved.")



## 7. Business Interpretation

The preferred model should not be selected using accuracy alone. In flight-delay prediction, **recall for delayed flights** is especially important because a false negative means the model predicts that a flight will be on time when it is actually delayed.

A model with higher recall can help the airline:
- Notify passengers earlier
- Reassign gates and crews
- Improve customer communication
- Reduce operational disruption
- Support proactive scheduling decisions

The final model should therefore balance recall, precision, F1-score, and ROC-AUC.


## 8. Optional: Load the Saved Model

In [ ]:

loaded_model = joblib.load("flight_delay_model.pkl")

# Example:
# sample_predictions = loaded_model.predict(X_test.head(5))
# print(sample_predictions)

print("Saved model loaded successfully.")
